# Set EM algorithm parameters, load packages, and load the dataset

In [ ]:
using Pkg
Pkg.activate(joinpath(dirname(dirname(@__DIR__)), "em-master"))  # Optional: activates local environment

# Optional: define data and code directory relative to current script
script_dir = dirname(dirname(@__DIR__))                      
em_dir = joinpath(script_dir, "em-master")  # EM algorithm folder
study_file = joinpath(script_dir, "Data", "Processed", "Study2.csv") #change to study1.csv for study1


# Set model and EM algorithm parameters
full = false               # Use diagonal covariance matrix if 'false'
emtol = 1e-3               # EM stopping threshold based on relative change
push!(LOAD_PATH, em_dir)  # Add directory for accessing custom files

# Load required packages
using EM, Distributions, LinearAlgebra, Optim, Statistics, Random
using GLM, DataFrames, NamedArrays, SpecialFunctions, CSV

# Load dataset
data = CSV.read(study_file, DataFrame)


# Rename 'subn' to 'sub'
rename!(data, :subn => :sub)

# Identify unique subjects and create a mapping to sequential IDs
unique_subjects = unique(data.sub)
subject_map = Dict(unique_subjects[i] => i for i in 1:length(unique_subjects))

# Apply the mapping to make 'sub' sequential as this is needed for EM
data.sub .= [subject_map[sub] for sub in data.sub]
# show data
data

# Model4 with Perseveration Parameters

In [ ]:


# EM variables
NS = 57                       # Number of subjects (set to 47 for study 1 and 57 for study 2)
subs = 1:NS                   # Subject indices
X = ones(Int, NS)             # design matrix specifying the group level model
startbetas = [0 0 0 0 0 0 0] # starting points for group level parameters
startsigma = [1, 1,1, 1, 1, 1, 1]          # Initial sigma(variance) values


# Model function to calculate negative log-likelihood
function M4Pers(params, data)
    
AS_T = params[1]
AD_T = params[2]  #Paired-ttest between conf_true and disconf_true
AS_F = params[3]
AD_F = params[4] #Paired-ttest between conf_false and disconf_false

nd_conf_true = 0.5 * (AS_T + AD_T)
Conf_True = 1 / (1 + exp(-nd_conf_true))

nd_disconf_true = 0.5 * (AS_T - AD_T)
Disconf_True = 1 / (1 + exp(-nd_disconf_true)) 

nd_conf_false = 0.5 * (AS_F + AD_F)
Conf_False = 1 / (1 + exp(-nd_conf_false))

nd_disconf_false = 0.5 * (AS_F - AD_F)
Disconf_False = 1 / (1 + exp(-nd_disconf_false)) 

Beta = 25 + 25 * erf(params[5] / sqrt(2))

tau = 1 / (1 + exp(-params[6]))   
    
    rho = 7*erf(params[7] / sqrt(2))
                            
     # Extract data columns (using multiple assignment)
    R, C1, C2, TrialType, FeedbackType, Condition = (
        data.Reward, data.C1, data.C2, data.TrialType, 
        data.FeedbackType, data.Condition
    )
    
    # Initialize Q-values matrix
     # Initialize likelihood
    lik = 0.0

    # Initialize Q values
    Q = zeros(typeof(Beta), 2, 4)    
    C = zeros(typeof(Beta), 2, 4)

    # Loop over trials
    for t in 1:length(C1)
        
        QC = Condition[t]

        chosen_idx = C1[t] == 1 ? 1 : 2
        unchosen_idx = 3 - chosen_idx
         Q_chosen = Q[chosen_idx, QC]
        Q_unchosen = Q[unchosen_idx, QC]

        # Increment likelihood
new_lik = log(1 / (1 + exp(-Beta * (Q[chosen_idx, QC] - Q[unchosen_idx, QC]) - rho * (C[chosen_idx, QC] - C[unchosen_idx, QC]))))
        
        # Check for infinite or NaN likelihood
        if isinf(new_lik) || isnan(new_lik)
            println("Infinite or NaN likelihood detected for subject $(sub[t]) at trial $t")
        end
        
        lik += new_lik
       ft = FeedbackType[t]

        if ft == 1
        PEC = R[t] - Q[chosen_idx, QC]
        elseif ft == 0
        PEU = R[t] - Q[unchosen_idx,QC]
        end
        tt = TrialType[t]
        c_t = [C1[t], C2[t]]
        

options = [chosen_idx, unchosen_idx]
for option in options
    pe = (ft == 1) ? PEC : PEU

    lr_factual = (tt == 1) ? ((pe > 0) ? Conf_True : Disconf_True) : ((pe > 0) ? Conf_False : Disconf_False)
    lr_C_factual = (tt == 1) ? ((pe > 0) ? Disconf_True : Conf_True) : ((pe > 0) ? Disconf_False : Conf_False)
    lr = (ft == 1) ? lr_factual : lr_C_factual

    if (ft == 1 && c_t[option] == 1) || (ft == 0 && c_t[3-option] == 1)
        Q[option, QC] = (1 - lr) * Q[option, QC] + lr * R[t]
    end
end
        
        C[chosen_idx, QC] = C[chosen_idx, QC] + tau * (1 - C[chosen_idx, QC])
        C[unchosen_idx, QC] = C[unchosen_idx, QC] + tau * (0-C[unchosen_idx, QC])

    end

    # Return minus log-likelihood
    return -lik
end



                        


## Run em to get best fit parameters for each subject

In [ ]:
# run em
# x contains the parameters for each subject (note not the same as variable X)
# l and h are per-subject likelihood and hessians

(betas, sigma, x, l, h) = em(data, subs, X, startbetas, startsigma, M4Pers, maxiter=2000, emtol=emtol, full=full)

Run hierarchical t-test using hierarchical standard errors

In [ ]:
using Distributions

# Calculate hierarchical standard errors
standarderrors = emerrors(data, subs, x, X, h, betas, sigma, M4Pers)

function em_ttest(betas, standarderrors, m; d=nothing, nsubjects=nothing)
    # Ensure betas is a vector for consistency
    betas = vec(betas)
    
    # Default to testing all parameters if `d` is not provided
    if d === nothing
        d = 1:length(betas)
    end
    
    # Ensure the number of means `m` matches the number of parameters being tested
    if length(m) != length(d)
        error("Mismatch: m has $(length(m)) elements, but $(length(d)) parameters are being tested.")
    end
    
    # Extract standard errors (assuming it's the first element in the tuple)
    se = standarderrors[1]
    
    # Extract betas and standard errors for the selected indices
    a = betas[d]  # Betas for the selected parameters
    e = se[d]     # Standard errors for the selected parameters
    
    # Calculate degrees of freedom (nsubjects - 1)
    df = nsubjects - 1
    
    # Calculate t-statistics for each parameter
    t = (a .- m) ./ e
    
    # Calculate two-tailed p-values for each t-statistic
    p = 2 * [ccdf(TDist(df), abs(ti)) for ti in t]
    
    # Create a named tuple with the results
    stats = (tstat = t, pval = p, df = df)
    
    # Return the statistics as the result
    return stats
end


In [ ]:
# Define the null hypothesis values
m = zeros(2)  # Null hypothesis values, testing against 0 for two parameters

# Specify the indices of parameters to test
d = [2, 4]  # Only test the second (TRUE) and fourth (FALSE) parameters 

# Perform the t-test
stats = em_ttest(betas, standarderrors, m; d=d, nsubjects=length(subs))

println(stats)

# The following function transforms back the parameters to their original value and is used to for calculating individual points, the mean, standard error of the mean, and confidence interval

In [ ]:
using SpecialFunctions  # Required for the erf function
using DataFrames
using CSV
using LinearAlgebra  # For Matrix conversion
using Statistics  # For std function

(standarderrors, pvalues, covmtx) = emerrors(data, subs, x, X, h, betas, sigma, M4Pers)
se = reshape(standarderrors, 1, length(betas))

function transform_back(x::Matrix)
    # Extract parameters directly from columns

    AS_T = x[:, 1]
    AD_T = x[:, 2]
    AS_F = x[:, 3]
    AD_F = x[:, 4]
    params5 = x[:, 5]
    params6 = x[:, 6]
    params7 = x[:, 7]

    nd_conf_true = 0.5 .* (AS_T .+ AD_T)
    nd_disconf_true = 0.5 .* (AS_T .- AD_T)
    nd_conf_false = 0.5 .* (AS_F .+ AD_F)
    nd_disconf_false = 0.5 .* (AS_F .- AD_F)
    
    # Transform the parameters


    Conf_True = 1.0 ./ (1.0 .+ exp.(-nd_conf_true))
    Conf_False = 1.0 ./ (1.0 .+ exp.(-nd_conf_false))
    Disconf_True = 1.0 ./ (1.0 .+ exp.(-nd_disconf_true))
    Disconf_False = 1.0 ./ (1.0 .+ exp.(-nd_disconf_false))


Beta = 25 .+ 25 .* erf.(((1 / sqrt(2)) .* params5))

    tau = 1.0 ./ (1.0 .+ exp.(-params6))

    rho = 7 .* erf.(params7 ./ sqrt(2))

    
    # Return DataFrame with transformed parameters
    return DataFrame(
        Conf_True = Conf_True ,
        Disconf_True = Disconf_True,
        Conf_False = Conf_False,
        Disconf_False = Disconf_False,
        Beta = Beta,
    tau = tau,
    rho = rho
    )
end


# Calculate SEM directly from individual subject data
betas_matrix = Matrix(reshape(betas, 1, length(betas)))
se_matrix = Matrix(se)

# Calculate 95% Confidence Interval bounds
xh_95ci = betas .+ 1.96 .* se
xl_95ci = betas .- 1.96 .* se

# Calculate SEM from CI
sem = (xh_95ci - xl_95ci) ./ (2 * 1.96)

# Transform the point estimates, CI bounds, and SEM bounds
df_point = transform_back(betas_matrix)
df_ci_upper = transform_back(xh_95ci)
df_ci_lower = transform_back(xl_95ci)
df_sem_upper = transform_back(betas_matrix .+ sem)
df_sem_lower = transform_back(betas_matrix .- sem)

# Create the final DataFrame
final_df = DataFrame()
for col in names(df_point)
    final_df[!, "$(col)_point"] = df_point[!, col]
    final_df[!, "$(col)_ci_lower"] = df_ci_lower[!, col]
    final_df[!, "$(col)_ci_upper"] = df_ci_upper[!, col]
    final_df[!, "$(col)_sem_lower"] = df_sem_lower[!, col]
    final_df[!, "$(col)_sem_upper"] = df_sem_upper[!, col]
end

# Display the final DataFrame
println(final_df)
CSV.write("M4PersStats.csv", final_df)


## Use the same function to create the estimates for each subject 

In [ ]:
df = transform_back(x)
CSV.write("M4PersPoints.csv", df)
